# Zadanie 8 - regresja logistyczna na GPU (CUDA)
Notatnik do odpalenia na Google Colab z GPU T4.

In [ ]:
# Sprawdzenie karty graficznej
!nvidia-smi

In [ ]:
import math
import time
import numpy as np
from numba import cuda

print("CUDA dostepne:", cuda.is_available())
if cuda.is_available():
    print("Karta:", cuda.get_current_device().name)

In [ ]:
# Kernele cuda do liczenia predykcji, gradientu i aktualizacji wag

@cuda.jit
def compute_errors_cuda_kernel(d_X, d_y, d_weights, d_errors, batch_start, batch_size, n_features):
    # Kazdy watek liczy jedna probke z batcha
    i = cuda.grid(1)
    if i < batch_size:
        row = batch_start + i
        z = 0.0
        for j in range(n_features):
            z += d_X[row, j] * d_weights[j]

        # Zabezpieczenie przed overflowem
        if z < -85.0:
            z_safe = -85.0
        elif z > 85.0:
            z_safe = 85.0
        else:
            z_safe = z

        pred = 1.0 / (1.0 + math.exp(-z_safe))
        d_errors[i] = pred - d_y[row]

@cuda.jit
def compute_gradient_cuda_kernel(d_X, d_errors, d_grad, batch_start, batch_size, n_features):
    # Kazdy watek liczy gradient dla jednej cechy j
    j = cuda.grid(1)
    if j < n_features:
        acc = 0.0
        for i in range(batch_size):
            acc += d_errors[i] * d_X[batch_start + i, j]
        d_grad[j] = acc / batch_size

@cuda.jit
def update_weights_cuda_kernel(d_weights, d_grad, lr, n_features):
    # Aktualizacja wag bezposrednio w pamieci GPU
    j = cuda.grid(1)
    if j < n_features:
        d_weights[j] -= lr * d_grad[j]

In [ ]:
def train_cuda_sgd(X_train, y_train, epochs=15, batch_size=256, lr=0.05, threads_per_block=128):
    n_samples, n_features = X_train.shape
    
    # Dane wrzucamy do pamieci GPU tylko raz na poczatku
    t0 = time.time()
    d_X = cuda.to_device(np.ascontiguousarray(X_train, dtype=np.float32))
    d_y = cuda.to_device(np.ascontiguousarray(y_train, dtype=np.float32))
    
    d_weights = cuda.to_device(np.zeros(n_features, dtype=np.float32))
    d_errors = cuda.device_array(batch_size, dtype=np.float32)
    d_grad = cuda.device_array(n_features, dtype=np.float32)
    print(f"Dane przeslane do VRAM w {time.time() - t0:.2f}s")

    blocks_features = (n_features + threads_per_block - 1) // threads_per_block
    times = []

    for epoch in range(1, epochs + 1):
        t_ep = time.time()
        
        for start in range(0, n_samples, batch_size):
            bs = min(batch_size, n_samples - start)
            if bs <= 0:
                continue

            blocks_err = (bs + threads_per_block - 1) // threads_per_block

            # Odpalamy obliczenia na GPU (bez przesylania danych przez PCIe)
            compute_errors_cuda_kernel[blocks_err, threads_per_block](
                d_X, d_y, d_weights, d_errors, start, bs, n_features
            )
            compute_gradient_cuda_kernel[blocks_features, threads_per_block](
                d_X, d_errors, d_grad, start, bs, n_features
            )
            update_weights_cuda_kernel[blocks_features, threads_per_block](
                d_weights, d_grad, lr, n_features
            )

        elapsed = time.time() - t_ep
        times.append(elapsed)
        print(f"Epoka {epoch:02d} (GPU) - czas: {elapsed:.3f}s")

    # Zwracamy wagi z GPU do CPU
    return d_weights.copy_to_host(), times

In [ ]:
# Generujemy przykladowe dane (50k probek, 32 cechy)
np.random.seed(42)
N, D = 50000, 32
X_raw = np.random.randn(N, D).astype(np.float32)
y_raw = (X_raw[:, 0] * 1.5 - X_raw[:, 1] * 0.8 > 0.0).astype(np.float32)

# Kolumna z jedynkami na bias
X_data = np.hstack([np.ones((N, 1), dtype=np.float32), X_raw])

print(f"Dane: {X_data.shape[0]} probek, {X_data.shape[1]} cech")

# 1. Trening na GPU
print("\n--- Trening GPU (CUDA) ---")
w_gpu, times_gpu = train_cuda_sgd(X_data, y_raw, epochs=10, batch_size=256, lr=0.05)
time_gpu = sum(times_gpu)

# 2. Trening na CPU do porownania
print("\n--- Trening CPU (sekwencyjny) ---")
def train_cpu(X, y, epochs=10, batch_size=256, lr=0.05):
    w = np.zeros(X.shape[1], dtype=np.float32)
    t0 = time.time()
    for _ in range(epochs):
        for start in range(0, X.shape[0], batch_size):
            xb = X[start:start+batch_size]
            yb = y[start:start+batch_size]
            pred = 1.0 / (1.0 + np.exp(-np.clip(xb @ w, -85.0, 85.0)))
            grad = (xb.T @ (pred - yb)) / xb.shape[0]
            w -= lr * grad
    return w, time.time() - t0

w_cpu, time_cpu = train_cpu(X_data, y_raw, epochs=10, batch_size=256, lr=0.05)

# Podsumowanie
print(f"\nLaczny czas GPU: {time_gpu:.3f}s")
print(f"Laczny czas CPU: {time_cpu:.3f}s")
print(f"Przyspieszenie GPU: {time_cpu / time_gpu:.2f}x")